# 🌋 VONA Data Scraper - Filtered by Volcano

## 📝 Deskripsi
Scraper ini mengumpulkan data **VONA (Volcano Observatory Notice for Aviation)** dari 5 gunung berapi spesifik di Indonesia:

1. **Merapi** (MER)
2. **Semeru** (SMR)
3. **Bromo** (BRO)
4. **Slamet** (SLA)
5. **Raung** (RAU)

## 📦 Data yang Dikumpulkan

### Data Lengkap dengan Detail:
- ⏰ Timestamp
- 🚦 Alert Level (Green/Yellow/Orange/Red)
- 🌋 **Volcano Name & Code**
- 📍 **Koordinat (Latitude, Longitude)** → untuk peta Leaflet.js
- 🏔️ **Summit Elevation (FT & M)** → untuk kalkulasi
- ✈️ **Volcanic Cloud Height** → **DATA PALING PENTING untuk AI model**
- 🌤️ **Remarks** → info cuaca & arah angin
- 📋 Notice Number, Area, Current/Previous Color Code

---

## 1. Import Libraries

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re 

## 2. Konfigurasi Gunung Berapi yang Akan Di-scrape

In [2]:
# Konfigurasi gunung berapi yang akan di-scrape
VOLCANOES = {
    # 'MER': 'Merapi',
    'SMR': 'Semeru',
    # 'BRO': 'Bromo',
    # 'SLA': 'Slamet',
    # 'RAU': 'Raung'
}

print("🌋 Gunung Berapi yang Akan Di-scrape:")
print("="*50)
for code, name in VOLCANOES.items():
    print(f"  - {name} (Code: {code})")
    print(f"    URL: https://magma.esdm.go.id/v1/vona?code={code}")
print("="*50)

🌋 Gunung Berapi yang Akan Di-scrape:
  - Semeru (Code: SMR)
    URL: https://magma.esdm.go.id/v1/vona?code=SMR


## 3. Fungsi untuk Scraping Detail VONA

In [3]:
def scrape_vona_detail(detail_url, headers):
    """
    Scrape informasi detail dari halaman VONA individual
    Mengambil data penting untuk model AI dan aplikasi
    """
    try:
        response = requests.get(detail_url, headers=headers, timeout=15)
        if response.status_code != 200:
            return None
        
        soup = BeautifulSoup(response.text, 'html.parser')
        table = soup.find('table', class_='table-mg-b')
        
        if not table:
            return None
        
        # Inisialisasi data detail
        detail_data = {
            'Volcano_Name': 'N/A',
            'Volcano_Code': 'N/A',
            'Issued_Time': 'N/A',
            'Current_Color_Code': 'N/A',
            'Previous_Color_Code': 'N/A',
            'Notice_Number': 'N/A',
            'Volcano_Location': 'N/A',
            'Latitude': None,
            'Longitude': None,
            'Area': 'N/A',
            'Summit_Elevation_FT': None,
            'Summit_Elevation_M': None,
            'Volcanic_Activity_Summary': 'N/A',
            'Volcanic_Cloud_Height': 'N/A',
            'Other_Volcanic_Cloud_Info': 'N/A',
            'Remarks': 'N/A'
        }
        
        # Parse semua row dalam tabel
        rows = table.find_all('tr')
        
        for row in rows:
            cells = row.find_all('td')
            if len(cells) < 3:
                continue
            
            row_num = cells[0].get_text(strip=True)
            
            # (2) Issued Time
            if row_num == '(2)':
                detail_data['Issued_Time'] = cells[3].get_text(strip=True)
            
            # (3) Volcano Name and Code
            elif row_num == '(3)':
                volcano_text = cells[3].get_text(strip=True)
                # Parse "Kaba (261220)" -> nama: Kaba, kode: 261220
                match = re.match(r'(.+?)\s*\((\d+)\)', volcano_text)
                if match:
                    detail_data['Volcano_Name'] = match.group(1).strip()
                    detail_data['Volcano_Code'] = match.group(2).strip()
                else:
                    detail_data['Volcano_Name'] = volcano_text
            
            # (4) Current Aviation Colour Code
            elif row_num == '(4)':
                detail_data['Current_Color_Code'] = cells[3].get_text(strip=True)
            
            # (5) Previous Aviation Colour Code
            elif row_num == '(5)':
                detail_data['Previous_Color_Code'] = cells[3].get_text(strip=True)
            
            # (7) Notice Number
            elif row_num == '(7)':
                detail_data['Notice_Number'] = cells[3].get_text(strip=True)
            
            # (8) Volcano Location - PENTING untuk koordinat
            elif row_num == '(8)':
                location_text = cells[3].get_text(strip=True)
                detail_data['Volcano_Location'] = location_text
                
                # Parse koordinat: "S 03 deg 31 min 12 sec E 102 deg 37 min 12 sec"
                lat_match = re.search(r'([NS])\s*(\d+)\s*deg\s*(\d+)\s*min\s*(\d+)\s*sec', location_text)
                
                if lat_match:
                    lat_deg = int(lat_match.group(2))
                    lat_min = int(lat_match.group(3))
                    lat_sec = int(lat_match.group(4))
                    lat = lat_deg + lat_min/60 + lat_sec/3600
                    if lat_match.group(1) == 'S':
                        lat = -lat
                    detail_data['Latitude'] = lat
                
                # Parse longitude dari full text
                lon_match = re.search(r'([EW])\s*(\d+)\s*deg\s*(\d+)\s*min\s*(\d+)\s*sec', location_text)
                if lon_match:
                    lon_deg = int(lon_match.group(2))
                    lon_min = int(lon_match.group(3))
                    lon_sec = int(lon_match.group(4))
                    lon = lon_deg + lon_min/60 + lon_sec/3600
                    if lon_match.group(1) == 'W':
                        lon = -lon
                    detail_data['Longitude'] = lon
            
            # (9) Area
            elif row_num == '(9)':
                detail_data['Area'] = cells[3].get_text(strip=True)
            
            # (10) Summit Elevation - PENTING untuk kalkulasi ketinggian
            elif row_num == '(10)':
                elevation_text = cells[3].get_text(strip=True)
                # Parse "6246 FT (1952 M)"
                ft_match = re.search(r'(\d+)\s*FT', elevation_text)
                m_match = re.search(r'(\d+)\s*M', elevation_text)
                
                if ft_match:
                    detail_data['Summit_Elevation_FT'] = int(ft_match.group(1))
                if m_match:
                    detail_data['Summit_Elevation_M'] = int(m_match.group(1))
            
            # (11) Volcanic Activity Summary
            elif row_num == '(11)':
                detail_data['Volcanic_Activity_Summary'] = cells[3].get_text(strip=True)
            
            # (12) Volcanic Cloud Height - SANGAT PENTING untuk model AI
            elif row_num == '(12)':
                detail_data['Volcanic_Cloud_Height'] = cells[3].get_text(strip=True)
            
            # (13) Other Volcanic Cloud Information
            elif row_num == '(13)':
                detail_data['Other_Volcanic_Cloud_Info'] = cells[3].get_text(strip=True)
            
            # (14) Remarks - PENTING untuk info cuaca dan arah angin
            elif row_num == '(14)':
                detail_data['Remarks'] = cells[3].get_text(strip=True)
        
        return detail_data
        
    except Exception as e:
        print(f"    Error scraping detail: {e}")
        return None

## 4. Fungsi untuk Scraping Berdasarkan Kode Gunung

In [4]:
import os

def scrape_vona_by_code(volcano_code, volcano_name, pages=5, include_details=True, 
                        save_every=5, csv_filename="data/vona_reports_filtered_complete.csv"):
    """
    Scrape data VONA berdasarkan kode gunung spesifik
    Menyimpan ke CSV setiap 'save_every' halaman untuk menghindari kehilangan data.
    
    Parameters:
    - volcano_code: kode gunung (MER, SMR, BRO, SLA, RAU)
    - volcano_name: nama gunung (untuk display)
    - pages: jumlah halaman yang akan di-scrape
    - include_details: True untuk scrape detail dari setiap link
    - save_every: simpan ke CSV setiap N halaman (default: 5)
    - csv_filename: nama file CSV output
    """
    base_url = f"https://magma.esdm.go.id/v1/vona?code={volcano_code}"
    all_data = []
    batch_data = []  # Data yang belum disimpan ke CSV
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    # Cek apakah file CSV sudah ada (untuk mode append)
    file_exists = os.path.exists(csv_filename)

    print(f"\n{'='*70}")
    print(f"🌋 Scraping: {volcano_name} (Code: {volcano_code})")
    print(f"   Total halaman: {pages} | Simpan setiap: {save_every} halaman")
    print(f"   Output: {csv_filename}")
    if file_exists:
        existing_df = pd.read_csv(csv_filename)
        print(f"   📂 File CSV sudah ada dengan {len(existing_df)} records, akan di-append")
    print(f"{'='*70}")

    for page in range(1, pages + 1):
        try:
            # Mengambil halaman list
            url = f"{base_url}&page={page}" if page > 1 else base_url
            response = requests.get(url, headers=headers, timeout=15)
            
            if response.status_code != 200:
                print(f"  ✗ Gagal mengakses halaman {page}: Status {response.status_code}")
                # Simpan sisa data sebelum berhenti
                if batch_data:
                    _save_batch_to_csv(batch_data, csv_filename)
                    batch_data = []
                break
                
            soup = BeautifulSoup(response.text, 'html.parser')
            reports = soup.find_all('div', class_='timeline-item')
            
            # Filter out timeline-day items
            actual_reports = [r for r in reports if 'timeline-day' not in r.get('class', [])]
            
            if not actual_reports:
                print(f"  ℹ️  Halaman {page} tidak ada data lagi. Berhenti.")
                # Simpan sisa data sebelum berhenti
                if batch_data:
                    _save_batch_to_csv(batch_data, csv_filename)
                    batch_data = []
                break
            
            print(f"  Halaman {page}: Ditemukan {len(actual_reports)} laporan")
            report_count = 0

            for idx, report in enumerate(actual_reports, 1):
                try:
                    # Ekstrak data dari list page
                    timeline_time = report.find('div', class_='timeline-time')
                    if not timeline_time:
                        continue
                    
                    timestamp_elem = timeline_time.find('small')
                    timestamp = timestamp_elem.get_text(strip=True) if timestamp_elem else 'N/A'
                    
                    alert_button = timeline_time.find('a', class_='btn')
                    alert_level = alert_button.get_text(strip=True) if alert_button else 'N/A'
                    
                    timeline_body = report.find('div', class_='timeline-body')
                    if not timeline_body:
                        continue
                    
                    title_elem = timeline_body.find('p', class_='timeline-title')
                    title = title_elem.get_text(strip=True) if title_elem else 'N/A'
                    
                    author_elem = timeline_body.find('p', class_='timeline-author')
                    author_info = author_elem.get_text(strip=True) if author_elem else 'N/A'
                    
                    text_elem = timeline_body.find('p', class_='timeline-text')
                    description = text_elem.get_text(strip=True) if text_elem else 'N/A'
                    
                    link_elem = timeline_body.find('a', class_='card-link')
                    detail_link = link_elem.get('href') if link_elem else 'N/A'
                    
                    # Data dasar
                    data_row = {
                        "Volcano_Filter": volcano_name,
                        "Volcano_Code_Filter": volcano_code,
                        "Timestamp": timestamp,
                        "Alert_Level": alert_level,
                        "Volcano_Report": title,
                        "Author_Observatory": author_info,
                        "Description": description,
                        "Detail_Link": detail_link,
                        "Page": page
                    }
                    
                    # Scrape detail jika diminta
                    if include_details and detail_link != 'N/A':
                        detail_data = scrape_vona_detail(detail_link, headers)
                        
                        if detail_data:
                            data_row.update(detail_data)
                        
                        # Delay untuk menghindari rate limiting
                        time.sleep(0.5)
                    
                    all_data.append(data_row)
                    batch_data.append(data_row)
                    report_count += 1
                    
                except Exception as e:
                    print(f"  ✗ Error parsing report {idx}: {e}")
                    continue
            
            print(f"  ✓ Halaman {page} selesai. Data terkumpul: {report_count} records")
            
            # 💾 Simpan ke CSV setiap 'save_every' halaman
            if page % save_every == 0 and batch_data:
                _save_batch_to_csv(batch_data, csv_filename)
                print(f"  💾 TERSIMPAN! Total di CSV sekarang: {_count_csv_rows(csv_filename)} records")
                batch_data = []
            
            # Delay antar halaman
            if page < pages:
                time.sleep(1.5)
                
        except Exception as e:
            print(f"  ✗ Error pada halaman {page}: {e}")
            # Simpan sisa data sebelum berhenti karena error
            if batch_data:
                _save_batch_to_csv(batch_data, csv_filename)
                print(f"  💾 Data darurat tersimpan! Total di CSV: {_count_csv_rows(csv_filename)} records")
                batch_data = []
            break

    # Simpan sisa data yang belum tersimpan (halaman terakhir yang tidak kelipatan save_every)
    if batch_data:
        _save_batch_to_csv(batch_data, csv_filename)
        print(f"  💾 Sisa data tersimpan! Total di CSV: {_count_csv_rows(csv_filename)} records")

    print(f"  📊 Total data {volcano_name} sesi ini: {len(all_data)} records")
    return all_data


def _save_batch_to_csv(batch_data, csv_filename):
    """Simpan batch data ke CSV (append jika file sudah ada)"""
    df_batch = pd.DataFrame(batch_data)
    file_exists = os.path.exists(csv_filename)
    df_batch.to_csv(csv_filename, mode='a', header=not file_exists, index=False, encoding='utf-8')


def _count_csv_rows(csv_filename):
    """Hitung jumlah baris di file CSV"""
    try:
        df_temp = pd.read_csv(csv_filename)
        return len(df_temp)
    except Exception:
        return 0

## 5. Scraping Semua Gunung Berapi Target

In [5]:
# Scraping semua gunung berapi yang ditargetkan
# Data disimpan ke CSV setiap 5 halaman untuk menghindari kehilangan data

CSV_FILENAME = "data/vona_reports_filtered_complete_v2.csv"
PAGES_PER_VOLCANO = 9999  # Batas maksimum, akan berhenti otomatis saat data habis
SAVE_EVERY = 5            # Simpan ke CSV setiap N halaman

print("\n" + "="*70)
print("🌋 MEMULAI SCRAPING DATA VONA LENGKAP")
print("="*70)
print(f"Target: {len(VOLCANOES)} gunung berapi")
print(f"Mode: Scraping semua halaman sampai habis")
print(f"Mode: Incremental Save setiap {SAVE_EVERY} halaman")
print(f"Output: {CSV_FILENAME}")
print("⏱️  Proses ini akan memakan waktu...\n")

# Hapus file CSV lama jika ingin mulai fresh (uncomment baris di bawah)
# import os; os.remove(CSV_FILENAME) if os.path.exists(CSV_FILENAME) else None

all_volcano_data = []

try:
    # Scrape setiap gunung
    for code, name in VOLCANOES.items():
        volcano_data = scrape_vona_by_code(
            volcano_code=code,
            volcano_name=name,
            pages=PAGES_PER_VOLCANO,
            include_details=True,
            save_every=SAVE_EVERY,
            csv_filename=CSV_FILENAME
        )
        
        all_volcano_data.extend(volcano_data)
        
        # Delay antar gunung untuk menghindari rate limiting
        time.sleep(2)
    
    # Baca file CSV final untuk preview
    df_filtered_complete = pd.read_csv(CSV_FILENAME)
    
    print(f"\n{'='*70}")
    print("✓ SCRAPING SELESAI!")
    print(f"{'='*70}")
    print(f"Total data di CSV: {len(df_filtered_complete)} records")
    print(f"Total kolom: {len(df_filtered_complete.columns)}")
    
    if not df_filtered_complete.empty:
        print(f"\n📊 Distribusi per Gunung:")
        print(df_filtered_complete['Volcano_Filter'].value_counts())
        
        print(f"\n📋 Kolom yang Tersedia:")
        for i, col in enumerate(df_filtered_complete.columns, 1):
            print(f"  {i}. {col}")
        
        # Tampilkan sample data
        print(f"\n📄 Sample Data (3 baris pertama):")
        important_cols = ['Volcano_Filter', 'Volcano_Name', 'Alert_Level', 
                         'Volcanic_Cloud_Height', 'Latitude', 'Longitude']
        available_cols = [col for col in important_cols if col in df_filtered_complete.columns]
        
        if available_cols:
            print(df_filtered_complete[available_cols].head(3))
    else:
        print("\n⚠️ DataFrame kosong!")

except Exception as e:
    print(f"\n✗ Error during scraping: {e}")
    import traceback
    traceback.print_exc()
    # Meskipun error, data yang sudah terscrape tetap aman di CSV
    if os.path.exists(CSV_FILENAME):
        df_saved = pd.read_csv(CSV_FILENAME)
        print(f"\n💾 Data yang sudah tersimpan di CSV: {len(df_saved)} records")
        print("   Data tidak hilang! Anda bisa melanjutkan scraping dari halaman terakhir.")


🌋 MEMULAI SCRAPING DATA VONA LENGKAP
Target: 1 gunung berapi
Mode: Scraping semua halaman sampai habis
Mode: Incremental Save setiap 5 halaman
Output: data/vona_reports_filtered_complete_v2.csv
⏱️  Proses ini akan memakan waktu...


🌋 Scraping: Semeru (Code: SMR)
   Total halaman: 9999 | Simpan setiap: 5 halaman
   Output: data/vona_reports_filtered_complete_v2.csv
  Halaman 1: Ditemukan 15 laporan
  ✓ Halaman 1 selesai. Data terkumpul: 15 records
  Halaman 2: Ditemukan 15 laporan
  ✓ Halaman 2 selesai. Data terkumpul: 15 records
  Halaman 3: Ditemukan 15 laporan
  ✓ Halaman 3 selesai. Data terkumpul: 15 records
  Halaman 4: Ditemukan 15 laporan
  ✓ Halaman 4 selesai. Data terkumpul: 15 records
  Halaman 5: Ditemukan 15 laporan
  ✓ Halaman 5 selesai. Data terkumpul: 15 records
  💾 TERSIMPAN! Total di CSV sekarang: 75 records
  Halaman 6: Ditemukan 15 laporan
  ✓ Halaman 6 selesai. Data terkumpul: 15 records
  Halaman 7: Ditemukan 15 laporan
  ✓ Halaman 7 selesai. Data terkumpul: 15 re

## 6. Simpan Data ke CSV

In [ ]:
# Simpan data ke CSV

if 'df_filtered_complete' in locals() and not df_filtered_complete.empty:
    filename = "data/vona_reports_filtered_complete.csv"
    
    try:
        df_filtered_complete.to_csv(filename, index=False, encoding='utf-8')
        
        print("="*70)
        print("✓ DATA BERHASIL DISIMPAN!")
        print("="*70)
        print(f"📁 File: {filename}")
        print(f"📊 Total records: {len(df_filtered_complete)}")
        print(f"📋 Total kolom: {len(df_filtered_complete.columns)}")
        
        print(f"\n🌋 Distribusi Data per Gunung:")
        volcano_counts = df_filtered_complete['Volcano_Filter'].value_counts()
        for volcano, count in volcano_counts.items():
            print(f"  - {volcano}: {count} records")
        
        print(f"\n💾 Data siap untuk digunakan dalam training model AI!")
        
    except Exception as e:
        print(f"✗ Error menyimpan file: {e}")
else:
    print("✗ Tidak ada data untuk disimpan")
    print("💡 Jalankan cell scraping terlebih dahulu!")

## 7. Analisis & Statistik Data

In [ ]:
# Analisis mendalam data yang sudah di-scrape

if 'df_filtered_complete' in locals() and not df_filtered_complete.empty:
    df = df_filtered_complete
    
    print("="*70)
    print("📊 STATISTIK DATA LENGKAP")
    print("="*70)
    print(f"\n✓ Total Records: {len(df)}")
    print(f"✓ Total Kolom: {len(df.columns)}")
    print(f"✓ Rentang Waktu: {df['Timestamp'].min()} - {df['Timestamp'].max()}")
    
    # Statistik per Gunung
    print(f"\n{'='*70}")
    print("🌋 DISTRIBUSI PER GUNUNG BERAPI")
    print(f"{'='*70}")
    volcano_stats = df.groupby('Volcano_Filter').agg({
        'Volcano_Report': 'count',
        'Alert_Level': lambda x: x.value_counts().index[0] if len(x) > 0 else 'N/A'
    }).rename(columns={'Volcano_Report': 'Total_Reports', 'Alert_Level': 'Most_Common_Alert'})
    print(volcano_stats)
    
    # Statistik per Alert Level
    if 'Alert_Level' in df.columns:
        print(f"\n{'='*70}")
        print("🚦 DISTRIBUSI ALERT LEVEL")
        print(f"{'='*70}")
        alert_counts = df['Alert_Level'].value_counts()
        for alert, count in alert_counts.items():
            percentage = (count / len(df) * 100)
            print(f"  {alert}: {count} records ({percentage:.1f}%)")
    
    # Statistik per Alert Level per Gunung
    print(f"\n{'='*70}")
    print("🌋 ALERT LEVEL PER GUNUNG")
    print(f"{'='*70}")
    alert_by_volcano = pd.crosstab(df['Volcano_Filter'], df['Alert_Level'])
    print(alert_by_volcano)
    
    # Data Quality Check
    print(f"\n{'='*70}")
    print("📋 KUALITAS DATA (Missing Values)")
    print(f"{'='*70}")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({
        'Missing_Count': missing,
        'Percentage': missing_pct
    })
    missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
    
    if len(missing_df) > 0:
        print(missing_df)
    else:
        print("✓ Tidak ada missing values!")
    
    # Statistik Koordinat (untuk validasi)
    if 'Latitude' in df.columns and 'Longitude' in df.columns:
        print(f"\n{'='*70}")
        print("📍 KOORDINAT GUNUNG BERAPI")
        print(f"{'='*70}")
        coord_stats = df.groupby('Volcano_Filter').agg({
            'Latitude': lambda x: x.dropna().iloc[0] if len(x.dropna()) > 0 else None,
            'Longitude': lambda x: x.dropna().iloc[0] if len(x.dropna()) > 0 else None
        })
        print(coord_stats)
    
    # Sample data untuk verifikasi
    print(f"\n{'='*70}")
    print("📄 SAMPLE DATA (2 records pertama per gunung)")
    print(f"{'='*70}")
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    
    for volcano in df['Volcano_Filter'].unique():
        print(f"\n🌋 {volcano}:")
        sample = df[df['Volcano_Filter'] == volcano].head(2)
        display_cols = ['Timestamp', 'Alert_Level', 'Volcanic_Cloud_Height', 
                       'Latitude', 'Longitude', 'Summit_Elevation_M']
        available_display_cols = [col for col in display_cols if col in sample.columns]
        print(sample[available_display_cols])
        
else:
    print("⚠️ Tidak ada data untuk dianalisis.")
    print("💡 Jalankan cell scraping terlebih dahulu!")

## 8. Export Info untuk Dokumentasi

In [ ]:
# Generate summary report

if 'df_filtered_complete' in locals() and not df_filtered_complete.empty:
    print("="*70)
    print("📝 RINGKASAN SCRAPING")
    print("="*70)
    print(f"\n✓ Total Data: {len(df_filtered_complete)} records")
    print(f"✓ Gunung Berapi: {df_filtered_complete['Volcano_Filter'].nunique()} gunung")
    print(f"✓ File Output: data/vona_reports_filtered_complete.csv")
    
    print(f"\n🌋 Detail per Gunung:")
    for code, name in VOLCANOES.items():
        count = len(df_filtered_complete[df_filtered_complete['Volcano_Code_Filter'] == code])
        print(f"  - {name} ({code}): {count} records")
    
    print(f"\n📊 Kolom Data yang Tersedia: {len(df_filtered_complete.columns)}")
    print(f"\n✅ Data siap digunakan untuk:")
    print(f"  1. Training Model Machine Learning")
    print(f"  2. Visualisasi Peta Interaktif")
    print(f"  3. Analisis Prediksi Sebaran Abu Vulkanik")
    print(f"  4. Integrasi dengan OpenWeatherMap API")
    
    print(f"\n{'='*70}")
    print("🎉 SCRAPING BERHASIL DISELESAIKAN!")
    print("="*70)
else:
    print("⚠️ Tidak ada data yang tersedia untuk ringkasan.")